# VELALAR COLLEGE OF ENGINEERING AND TECHNOLOGY (VCET), ERODE - 12
## DEPARTMENT OF INFORMATION TECHNOLOGY
### Course: 25ITC31 - Foundations of Data Science | Academic Year: 2026-27
---
## Project-Based Assessment: Scenario 3
### **College Canteen Ordering and Sales Analysis System**
**Register Numbers:** 3, 33, 63, 93

### Notebook Objectives:
1. **FR 7**: Retrieve sales transaction dataset from SQLite database / CSV.
2. Clean, handle missing values, and preprocess canteen ordering records.
3. Perform Exploratory Data Analysis (EDA) and compute descriptive statistics.
4. **FR 8**: Generate publication-quality visualizations for:
   - Item-wise sales volume
   - Daily revenue trends
   - Top 5 popular food items
   - Category-wise sales contribution

## 1. Import Required Libraries

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.autolayout'] = True

print("Libraries successfully imported!")

## 2. Retrieve Canteen Sales Dataset (FR 7)

In [ ]:
# Load data from CSV file or SQLite database
csv_path = 'canteen_sales_data.csv'
db_path = 'canteen_db.sqlite'

if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    query = """
    SELECT 
        o.order_id, o.order_date, o.order_time, c.customer_id, c.customer_name, 
        c.role AS customer_role, f.item_id, f.item_name, cat.category_name AS category, 
        oi.unit_price, oi.quantity, oi.line_total, p.payment_method, p.payment_status
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN food_items f ON oi.item_id = f.item_id
    JOIN categories cat ON f.category_id = cat.category_id
    LEFT JOIN payments p ON o.order_id = p.order_id
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    print("Retrieved records from SQLite database.")
else:
    df = pd.read_csv(csv_path)
    print(f"Loaded dataset from CSV: {csv_path}")

print(f"Total transaction records: {len(df)}")
df.head()

## 3. Data Cleaning and Preprocessing (25ITC31 Unit 4)

In [ ]:
# Inspect dataset info
print("=== Missing Values Check ===")
print(df.isnull().sum())

print("\n=== Duplicate Records Check ===")
duplicates = df.duplicated().sum()
print(f"Duplicates found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()

# Data type conversions
df['order_date'] = pd.to_datetime(df['order_date'])
df['unit_price'] = df['unit_price'].astype(float)
df['quantity'] = df['quantity'].astype(int)
df['line_total'] = df['line_total'].astype(float)

# Feature engineering: Extract day name and order hour
df['day_name'] = df['order_date'].dt.day_name()
df['hour'] = pd.to_datetime(df['order_time'], format='%H:%M:%S').dt.hour

print("\nData cleaning and type casting completed successfully!")

## 4. Descriptive Statistics & Summary (25ITC31 Unit 1 & 5)

In [ ]:
total_revenue = df['line_total'].sum()
unique_orders = df['order_id'].nunique()
total_items_sold = df['quantity'].sum()
avg_order_value = total_revenue / unique_orders

print("=== CANTEEN EXECUTIVE SUMMARY ===")
print(f"Total Revenue Generated : Rs. {total_revenue:,.2f}")
print(f"Total Orders Completed  : {unique_orders}")
print(f"Total Food Units Sold   : {total_items_sold}")
print(f"Average Order Value     : Rs. {avg_order_value:.2f}")

df[['unit_price', 'quantity', 'line_total']].describe().round(2)

## 5. Visualizations (FR 8)
### 5.1 Item-wise Sales Quantity

In [ ]:
plt.figure(figsize=(10, 6))
item_sales = df.groupby('item_name')['quantity'].sum().sort_values(ascending=True)
colors = sns.color_palette("viridis", len(item_sales))
bars = plt.barh(item_sales.index, item_sales.values, color=colors)
plt.title('Campus Canteen Item-wise Total Sales Volume (Units Sold)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Quantity Sold', fontsize=12)
plt.ylabel('Food Item Name', fontsize=12)
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.3, bar.get_y() + bar.get_height()/2, f'{int(width)}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

### 5.2 Daily Revenue Trend (Rs.)

In [ ]:
plt.figure(figsize=(9, 5))
daily_rev = df.groupby(df['order_date'].dt.strftime('%Y-%m-%d'))['line_total'].sum()
plt.plot(daily_rev.index, daily_rev.values, marker='o', color='#0284c7', linewidth=2.5, markersize=7)
plt.title('Daily Revenue Trend (Rs.) - Campus Canteen', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Order Date', fontsize=12)
plt.ylabel('Total Revenue (Rs.)', fontsize=12)
plt.xticks(rotation=30)
for x, y in zip(daily_rev.index, daily_rev.values):
    plt.annotate(f'Rs.{int(y)}', (x, y), textcoords="offset points", xytext=(0, 8), ha='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()

### 5.3 Top 5 Most Popular Food Items

In [ ]:
plt.figure(figsize=(8, 5))
top5 = df.groupby('item_name')['quantity'].sum().sort_values(ascending=False).head(5)
sns.barplot(x=top5.values, y=top5.index, palette="mako")
plt.title('Top 5 Most Popular Food Items (Units Sold)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Total Units Sold', fontsize=12)
plt.ylabel('Food Item', fontsize=12)
for idx, val in enumerate(top5.values):
    plt.text(val + 0.2, idx, f'{val} units', va='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.show()

### 5.4 Revenue Contribution by Category

In [ ]:
plt.figure(figsize=(7, 6))
cat_revenue = df.groupby('category')['line_total'].sum()
colors = ['#38bdf8', '#34d399', '#fbbf24', '#a78bfa']
plt.pie(cat_revenue, labels=cat_revenue.index, autopct='%1.1f%%', startangle=140, colors=colors,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
plt.title('Revenue Contribution by Food Category', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()